# Complete ML Assignment

This notebook covers:
1. Global Pollution Analysis and Energy Recovery
2. Food Delivery Time Prediction

Includes EDA, preprocessing, feature engineering, Linear Regression, Logistic Regression, evaluation metrics, visualizations, and reporting sections.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    r2_score, mean_squared_error, mean_absolute_error,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)

pollution_df = pd.read_csv("Global_Pollution_Analysis(3).csv")
food_df = pd.read_csv("Food_Delivery_Time_Prediction(2).csv")

print("Pollution Shape:", pollution_df.shape)
print("Food Shape:", food_df.shape)


## Part A - Global Pollution Analysis and Energy Recovery

In [ ]:

# Missing value handling
pollution_df = pollution_df.copy()

num_cols = pollution_df.select_dtypes(include=np.number).columns
pollution_df[num_cols] = SimpleImputer(strategy="median").fit_transform(pollution_df[num_cols])

# Encoding
le_country = LabelEncoder()
pollution_df["Country_Encoded"] = le_country.fit_transform(pollution_df["Country"])

# Scaling selected pollution indexes
scale_cols = ["Air_Pollution_Index","Water_Pollution_Index","Soil_Pollution_Index"]
pollution_df[scale_cols] = StandardScaler().fit_transform(pollution_df[scale_cols])

pollution_df.head()


In [ ]:

# Descriptive statistics
pollution_df.describe()


In [ ]:

# Correlation Heatmap
plt.figure(figsize=(10,6))
sns.heatmap(pollution_df.select_dtypes(include=np.number).corr(), cmap="coolwarm")
plt.title("Correlation Heatmap")
plt.show()


In [ ]:

# Visualizations
pollution_df.groupby("Year")["Air_Pollution_Index"].mean().plot(figsize=(10,4))
plt.title("Air Pollution Trend Over Years")
plt.show()

plt.figure(figsize=(8,4))
sns.boxplot(x=pollution_df["Air_Pollution_Index"])
plt.show()


In [ ]:

# Feature Engineering
pollution_df["Energy_Consumption_Per_Capita_Calc"] = (
    pollution_df["Energy_Consumption_Per_Capita (in MWh)"]
)

pollution_df["Year_Trend"] = pollution_df["Year"] - pollution_df["Year"].min()

pollution_df.head()


In [ ]:

# Linear Regression - Predict Energy Recovery

target = "Energy_Recovered (in GWh)"

features = [
    "Air_Pollution_Index",
    "CO2_Emissions (in MT)",
    "Industrial_Waste (in tons)",
    "Water_Pollution_Index",
    "Soil_Pollution_Index",
    "Country_Encoded"
]

X = pollution_df[features]
y = pollution_df[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr = LinearRegression()
lr.fit(X_train, y_train)

pred = lr.predict(X_test)

print("R2:", r2_score(y_test, pred))
print("MSE:", mean_squared_error(y_test, pred))
print("MAE:", mean_absolute_error(y_test, pred))


In [ ]:

# Logistic Regression - Pollution Severity Classification

pollution_df["Pollution_Severity"] = pd.qcut(
    pollution_df["Air_Pollution_Index"],
    q=3,
    labels=["Low","Medium","High"]
)

severity_encoder = LabelEncoder()
y = severity_encoder.fit_transform(pollution_df["Pollution_Severity"])

X = pollution_df[[
    "Air_Pollution_Index",
    "CO2_Emissions (in MT)",
    "Industrial_Waste (in tons)"
]]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

clf = LogisticRegression(max_iter=2000)
clf.fit(X_train,y_train)

pred = clf.predict(X_test)

print("Accuracy:", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred,average="weighted"))
print("Recall:", recall_score(y_test,pred,average="weighted"))
print("F1:", f1_score(y_test,pred,average="weighted"))
print(classification_report(y_test,pred))

cm = confusion_matrix(y_test,pred)
sns.heatmap(cm, annot=True, fmt="d")
plt.title("Confusion Matrix")
plt.show()


## Part B - Food Delivery Time Prediction

In [ ]:

food = food_df.copy()

cat_cols = food.select_dtypes(include="object").columns

for col in cat_cols:
    food[col] = LabelEncoder().fit_transform(food[col].astype(str))

food = pd.DataFrame(
    SimpleImputer(strategy="median").fit_transform(food),
    columns=food.columns
)

food.head()


In [ ]:

# Linear Regression - Predict Delivery Time

X = food.drop("Delivery_Time", axis=1)
y = food["Delivery_Time"]

X_train, X_test, y_train, y_test = train_test_split(
    X,y,test_size=0.2,random_state=42
)

model = LinearRegression()
model.fit(X_train,y_train)

pred = model.predict(X_test)

print("R2:", r2_score(y_test,pred))
print("MSE:", mean_squared_error(y_test,pred))
print("MAE:", mean_absolute_error(y_test,pred))


In [ ]:

# Logistic Regression - Fast vs Delayed Delivery

threshold = food["Delivery_Time"].median()
y_class = (food["Delivery_Time"] > threshold).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X,y_class,test_size=0.2,random_state=42
)

log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_train,y_train)

pred = log_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test,pred))
print("Precision:", precision_score(y_test,pred))
print("Recall:", recall_score(y_test,pred))
print("F1:", f1_score(y_test,pred))

cm = confusion_matrix(y_test,pred)
sns.heatmap(cm, annot=True, fmt="d")
plt.title("Food Delivery Confusion Matrix")
plt.show()



# Final Report

## Pollution Project
- Missing values handled.
- Pollution indices normalized.
- Country encoded.
- Correlation heatmap, trend analysis and boxplots included.
- Linear Regression predicts Energy Recovery.
- Logistic Regression classifies Low/Medium/High pollution severity.
- Metrics: R², MSE, MAE, Accuracy, Precision, Recall, F1 and Confusion Matrix.

## Food Delivery Project
- Data preprocessing completed.
- Linear Regression predicts Delivery Time.
- Logistic Regression classifies Fast vs Delayed deliveries.
- Evaluation metrics included.

## Actionable Insights
- Countries with higher pollution and industrial waste should invest in waste-to-energy systems.
- Renewable energy adoption can improve energy recovery outcomes.
- Traffic and weather conditions strongly influence delivery performance.
- Operational optimization can reduce delays and improve customer satisfaction.
